In [1]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
load_dotenv(".env")

c:\Users\kolan\OneDrive\Pulpit\ml\TabPFN-Distil\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from benchmark_datasets import OpenMLBenchmark

In [3]:
bench = OpenMLBenchmark()
print("Benchmark suite (20 datasets):")
print(bench.list().to_string(index=False))

ds = bench.load("monks-problems-2")
print(f"\nLoaded {ds.name!r}: X={ds.X.shape} ({ds.X.dtype}), y={ds.y.shape}")
print(f"classes: {ds.n_classes}, feature count: {len(ds.feature_names)}")

Benchmark suite (20 datasets):
             name  data_id           task  n_rows                                              note
      tic-tac-toe       50 classification     958           XOR-like win patterns, pure interaction
 monks-problems-2      334 classification     601                     synthetic XOR / parity target
            sonar       40 classification     208    60 correlated sonar bands, non-linear boundary
       ionosphere       59 classification     351                          radar signal, non-linear
          vehicle       54 classification     846                       4-class silhouette geometry
             wdbc     1510 classification     569    breast cancer, non-linear feature interactions
         diabetes       37 classification     768         Pima diabetes, classic non-linear medical
             ilpd     1480 classification     583          Indian liver patient, non-linear medical
    balance-scale       11 classification     625           target is

In [4]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.metrics import accuracy_score
sss = StratifiedShuffleSplit(1,test_size=0.8)
for i, (train_index, test_index) in enumerate(sss.split(ds.X, ds.y)):
    X_train = ds.X[train_index]
    y_train = ds.y[train_index]
    X_test = ds.X[test_index]
    y_test = ds.y[test_index]

    clf = TabPFNClassifier()
    clf.fit(X_train, y_train)
    base_pfn_pred = clf.predict(X_test)
    print('Base tabPFN acc: ', accuracy_score(y_test,base_pfn_pred))

    rfr = RandomForestClassifier(n_estimators=1)
    rfr.fit(X_train, y_train)
    base_rfr_pred = rfr.predict(X_test)
    print('Base RFR acc: ', accuracy_score(y_test,base_rfr_pred))
    
    X_aug = X_train
    y_aug = y_train
    for _ in range(3):
        aug_size = round(len(X_train)*1)
        rand_mask = np.random.rand(aug_size, len(X_train[0])) > 0.95
        rep_val = np.random.rand(aug_size, len(X_train[0])) > 0.5
        X_aug_new = X_train * (1-rand_mask) + rep_val*rand_mask

        y_aug_new = clf.predict(X_aug_new)

        X_aug = np.concat([X_aug, X_aug_new])
        y_aug = np.concat([y_aug, y_aug_new])

    rfr_aug = RandomForestClassifier(n_estimators=1)
    rfr_aug.fit(X_aug, y_aug)
    rfr_aug_pred = rfr_aug.predict(X_test)
    print('RFR aug acc: ', accuracy_score(y_test,rfr_aug_pred))


c:\Users\kolan\OneDrive\Pulpit\ml\TabPFN-Distil\.venv\Lib\site-packages\tabpfn\validation.py:142: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


Base tabPFN acc:  1.0
Base RFR acc:  0.768595041322314
RFR aug acc:  0.8512396694214877
